In [1]:
#Instalar las librerias
%pip install pandas numpy requests pyarrow sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Importamos libreria
import pandas as pd
import numpy as np
import unicodedata

In [ ]:
#Seteo endpoints
BASE_URL = "https://www.datos.gov.co/resource"

DATASETS = {
    "air_colombia": "g4t8-zkc3.csv",   # Calidad del Aire en Colombia
    "air_annual": "kekd-7v7h.csv",   # Calidad del Aire en Colombia (Promedio Anual)
    "vehicle": "u3vn-bdcy.csv"       # Crecimiento del Parque Automotor RUNT 2.0
}


In [4]:
#Extraemos los datos
def extraer_datos(endpoint, limit=None):

    if limit is not None:
        url = f"{BASE_URL}/{endpoint}?$limit={limit}"
    else:
        url = f"{BASE_URL}/{endpoint}"

    df = pd.read_csv(url)

    print(f"Endpoint: {endpoint}")
    print(f"Registros descargados: {len(df)}")

    return df

In [ ]:
#Funcion para normalizar los datos, setear en minusculas y normanizar con unicode
def normalizar_texto(texto):
    if pd.isna(texto):
        return None

    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    texto = " ".join(texto.split())
    return texto

In [ ]:
#Extraemos el data set air_colombia
df_air_raw = extraer_datos(DATASETS["air_colombia"], limit=1000000)

In [ ]:
#Extraemos el data set air_annual
df_air_annual_raw = extraer_datos(DATASETS["air_annual"], limit=29683)

In [ ]:
#Extraemos el data set vehicle
df_vehicle_raw = extraer_datos(DATASETS["vehicle"], limit=189247)

In [ ]:
print("Calidad del aire:")
display(df_air_raw.head())

print("Promedio anual:")
display(df_air_annual_raw.head())

print("Parque automotor:")
display(df_vehicle_raw.head())

In [ ]:
#Columnas dataset aire colombia
col_fecha_air = "med_fecha_inicio"
col_fecha_fin_air = "med_fecha_final"
col_ciudad_air = "municipio"
col_departamento_air = "departamento"
col_contaminante_air = "msfl_code"
col_valor_air = "med_concentracion_estandar"
col_estacion_air = "nombre_est"
col_estacion_id_air = "estacion_id"

#Columnas calidad anual del aire
col_estacion_id_air_annual = "id_estacion"
col_autoridad_air_annual = "autoridad_ambiental"
col_estacion_air_annual = "estaci_n"
col_variable_air_annual = "variable"
col_unidades_air_annual = "unidades"
col_anio_air_annual = "a_o"
col_promedio_air_annual = "promedio"
col_minimo_air_annual = "m_nimo"
col_departamento_air_annual = "nombre_del_departamento"
col_municipio_air_annual = "nombre_del_municipio"
col_tipo_estacion_air_annual = "tipo_de_estaci_n"

#Columnas parque automotor
col_departamento_vehicle = "nombre_departamento"
col_municipio_vehicle = "nombre_municipio"
col_servicio_vehicle = "nombre_servicio"
col_estado_vehicle = "estado_del_vehiculo"
col_clase_vehicle = "nombre_de_la_clase"
col_fecha_registro_vehicle = "fecha_de_registro"
col_cantidad_vehicle = "cantidad"
col_mes_pub_vehicle = "mes_de_publicacion"
col_anio_pub_vehicle = "a_o_de_publicacion"

In [ ]:
#limpiamos nuestro data set y creamos nuevos campos
df_air = df_air_raw.copy()
df_air.columns = df_air.columns.str.lower().str.strip()

df_air["fecha_inicio_etl"] = pd.to_datetime(df_air[col_fecha_air], errors="coerce")
df_air["fecha_fin_etl"] = pd.to_datetime(df_air[col_fecha_fin_air], errors="coerce")

df_air["ciudad_etl"] = df_air[col_ciudad_air].apply(normalizar_texto)
df_air["departamento_etl"] = df_air[col_departamento_air].apply(normalizar_texto)
df_air["contaminante_etl"] = df_air[col_contaminante_air].astype(str).apply(normalizar_texto)
df_air["valor_etl"] = pd.to_numeric(df_air[col_valor_air], errors="coerce")
df_air["estacion_etl"] = df_air[col_estacion_air].astype(str).apply(normalizar_texto)
df_air["estacion_id_etl"] = pd.to_numeric(df_air[col_estacion_id_air], errors="coerce")

df_air["anio_etl"] = df_air["fecha_inicio_etl"].dt.year
df_air["mes_etl"] = df_air["fecha_inicio_etl"].dt.month
df_air["dia_etl"] = df_air["fecha_inicio_etl"].dt.day
df_air["periodo_etl"] = df_air["fecha_inicio_etl"].dt.to_period("M").astype(str)

df_air = df_air[
    [
        "estacion_id_etl",
        "estacion_etl",
        "fecha_inicio_etl",
        "fecha_fin_etl",
        "anio_etl",
        "mes_etl",
        "dia_etl",
        "periodo_etl",
        "departamento_etl",
        "ciudad_etl",
        "contaminante_etl",
        "valor_etl"
    ]
].dropna(subset=["fecha_inicio_etl", "departamento_etl", "ciudad_etl", "contaminante_etl", "valor_etl"])

df_air.head()

In [ ]:
#Se crea una agregacion del data set df_air
df_air_agg = df_air.groupby(
    ["departamento_etl", "ciudad_etl", "periodo_etl", "contaminante_etl"],
    dropna=False
).agg(
    concentracion_promedio=("valor_etl", "mean"),
    concentracion_maxima=("valor_etl", "max"),
    concentracion_minima=("valor_etl", "min"),
    total_mediciones=("valor_etl", "count"),
    total_estaciones=("estacion_id_etl", "nunique")
).reset_index()

df_air_agg.head()

In [ ]:
df_air_pivot = df_air_agg.pivot_table(
    index=["departamento_etl", "ciudad_etl", "periodo_etl"],
    columns="contaminante_etl",
    values="concentracion_promedio",
    aggfunc="mean"
).reset_index()

df_air_pivot.columns.name = None
df_air_pivot.columns = [
    f"cont_{col}" if col not in ["departamento_etl", "ciudad_etl", "periodo_etl"] else col
    for col in df_air_pivot.columns
]

df_air_pivot.head()

In [ ]:
#limpiamos nuestro data set y creamos nuevos campos
df_air_annual = df_air_annual_raw.copy()
df_air_annual.columns = df_air_annual.columns.str.lower().str.strip()

df_air_annual["estacion_id_etl"] = pd.to_numeric(df_air_annual[col_estacion_id_air_annual], errors="coerce")
df_air_annual["autoridad_etl"] = df_air_annual[col_autoridad_air_annual].astype(str).apply(normalizar_texto)
df_air_annual["estacion_etl"] = df_air_annual[col_estacion_air_annual].astype(str).apply(normalizar_texto)
df_air_annual["variable_etl"] = df_air_annual[col_variable_air_annual].astype(str).apply(normalizar_texto)
df_air_annual["unidades_etl"] = df_air_annual[col_unidades_air_annual].astype(str).apply(normalizar_texto)

df_air_annual["anio_etl"] = pd.to_numeric(df_air_annual[col_anio_air_annual], errors="coerce")
df_air_annual["promedio_etl"] = pd.to_numeric(df_air_annual[col_promedio_air_annual], errors="coerce")
df_air_annual["minimo_etl"] = pd.to_numeric(df_air_annual[col_minimo_air_annual], errors="coerce")

df_air_annual["departamento_etl"] = df_air_annual[col_departamento_air_annual].apply(normalizar_texto)
df_air_annual["ciudad_etl"] = df_air_annual[col_municipio_air_annual].apply(normalizar_texto)
df_air_annual["tipo_estacion_etl"] = df_air_annual[col_tipo_estacion_air_annual].astype(str).apply(normalizar_texto)

df_air_annual = df_air_annual[
    [
        "estacion_id_etl",
        "autoridad_etl",
        "estacion_etl",
        "variable_etl",
        "unidades_etl",
        "anio_etl",
        "promedio_etl",
        "minimo_etl",
        "departamento_etl",
        "ciudad_etl",
        "tipo_estacion_etl"
    ]
].dropna(subset=["anio_etl", "departamento_etl", "ciudad_etl", "variable_etl", "promedio_etl"])

df_air_annual.head()

In [ ]:
df_air_annual_agg = df_air_annual.groupby(
    ["departamento_etl", "ciudad_etl", "anio_etl", "variable_etl"],
    dropna=False
).agg(
    promedio_anual=("promedio_etl", "mean"),
    minimo_anual=("minimo_etl", "mean"),
    total_estaciones_anuales=("estacion_id_etl", "nunique")
).reset_index()

df_air_annual_agg.head()

In [ ]:
df_air_annual_pivot = df_air_annual_agg.pivot_table(
    index=["departamento_etl", "ciudad_etl", "anio_etl"],
    columns="variable_etl",
    values="promedio_anual",
    aggfunc="mean"
).reset_index()

df_air_annual_pivot.columns.name = None
df_air_annual_pivot.columns = [
    f"anual_{col}" if col not in ["departamento_etl", "ciudad_etl", "anio_etl"] else col
    for col in df_air_annual_pivot.columns
]

df_air_annual_pivot.head()

In [ ]:
#limpiamos nuestro data set y creamos nuevos campos
df_vehicle = df_vehicle_raw.copy()
df_vehicle.columns = df_vehicle.columns.str.lower().str.strip()

df_vehicle["departamento_etl"] = df_vehicle[col_departamento_vehicle].apply(normalizar_texto)
df_vehicle["ciudad_etl"] = df_vehicle[col_municipio_vehicle].apply(normalizar_texto)
df_vehicle["servicio_etl"] = df_vehicle[col_servicio_vehicle].astype(str).apply(normalizar_texto)
df_vehicle["estado_vehiculo_etl"] = df_vehicle[col_estado_vehicle].astype(str).apply(normalizar_texto)
df_vehicle["clase_vehiculo_etl"] = df_vehicle[col_clase_vehicle].astype(str).apply(normalizar_texto)

df_vehicle["fecha_registro_etl"] = pd.to_datetime(df_vehicle[col_fecha_registro_vehicle], errors="coerce")
df_vehicle["cantidad_etl"] = pd.to_numeric(df_vehicle[col_cantidad_vehicle], errors="coerce")

# MAPEO DE MESES (CLAVE)
map_meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "setiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

df_vehicle["mes_etl"] = (
    df_vehicle[col_mes_pub_vehicle]
    .astype(str)
    .str.lower()
    .str.strip()
    .map(map_meses)
)

df_vehicle["anio_etl"] = pd.to_numeric(df_vehicle[col_anio_pub_vehicle], errors="coerce")

# crear periodo
df_vehicle["periodo_etl"] = (
    df_vehicle["anio_etl"].astype("Int64").astype(str) + "-" +
    df_vehicle["mes_etl"].astype("Int64").astype(str).str.zfill(2)
)

print("Antes de filtrar:", df_vehicle.shape)

df_vehicle = df_vehicle[
    [
        "departamento_etl",
        "ciudad_etl",
        "servicio_etl",
        "estado_vehiculo_etl",
        "clase_vehiculo_etl",
        "fecha_registro_etl",
        "cantidad_etl",
        "mes_etl",
        "anio_etl",
        "periodo_etl"
    ]
].dropna(subset=["departamento_etl", "ciudad_etl", "cantidad_etl", "mes_etl", "anio_etl"])

print("Después de filtrar:", df_vehicle.shape)

df_vehicle.head()

In [ ]:
df_vehicle_mensual = df_vehicle.groupby(
    ["departamento_etl", "ciudad_etl", "periodo_etl"],
    dropna=False
).agg(
    total_vehiculos=("cantidad_etl", "sum"),
    total_clases=("clase_vehiculo_etl", "nunique"),
    total_servicios=("servicio_etl", "nunique")
).reset_index()

df_vehicle_mensual.head()

In [ ]:
#Unimos las agregaciones df_air_pivot y df_vehicle_mensual
df_integrado_mensual = pd.merge(
    df_air_pivot,
    df_vehicle_mensual,
    on=["departamento_etl"],
    how="inner"
)



df_integrado_mensual["log_total_vehiculos"] = np.log1p(df_integrado_mensual["total_vehiculos"])

df_integrado_mensual.head()